In [1]:
!pip install langchain openai chromadb pypdf2 langchain-openai langchain-community -q
print("Installation complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68

In [2]:
import os
os.environ["OPENAI_API_KEY"] = "Your-API-Key"
print("API key set!")

API key set!


In [6]:
!pip install pypdf -q
print("pypdf installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.3 MB/s eta 0:00:00
pypdf installed!


In [7]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader('/content/Research_paper.pdf')
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print("\nFirst page preview:")
print(pages[0].page_content[:500])

Total pages loaded: 15

First page preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content[:300])

Total chunks: 52

Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Create embeddings
embeddings = OpenAIEmbeddings()

# Store chunks in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Stored {len(chunks)} chunks in ChromaDB!")

Stored 52 chunks in ChromaDB!


In [12]:
# Test similarity search
query = "What is the attention mechanism?"
results = vectorstore.similarity_search(query, k=3)

print(f"Top 3 relevant chunks for: '{query}'\n")
for i, doc in enumerate(results):
    print(f"Chunk {i+1}:")
    print(doc.page_content[:200])
    print("---")

Top 3 relevant chunks for: 'What is the attention mechanism?'

Chunk 1:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as 
---
Chunk 2:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need

---
Chunk 3:
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS
---


In [16]:
!pip install langchain -q

In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize GPT
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Build retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt template
template = """Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context: {context}
Question: {question}
Answer:"""

prompt = PromptTemplate.from_template(template)

# Build chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("QA Chain ready!")

QA Chain ready!


In [22]:
# Ask a question!
question = "What is the attention mechanism?"
answer = qa_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What is the attention mechanism?

Answer: An attention mechanism can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum.


In [23]:
questions = [
    "What is the transformer architecture?",
    "How many attention heads are used?",
    "What optimizer was used for training?",
    "What datasets were used to evaluate the model?"
]

for question in questions:
    answer = qa_chain.invoke(question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("---")

Q: What is the transformer architecture?
A: The Transformer architecture is based on stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a stack of N = 6 identical layers, each with two sub-layers - a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. The model employs residual connections and layer normalization to facilitate these connections. The output dimension of all sub-layers and embedding layers is dmodel = 512.
---
Q: How many attention heads are used?
A: 8 attention heads are used.
---
Q: What optimizer was used for training?
A: The Adam optimizer was used for training.
---
Q: What datasets were used to evaluate the model?
A: The WMT 2014 English-German dataset, the WMT 2014 English-French dataset, the Penn Treebank, the high-confidence and BerkleyParser corpora.
---
